In [ ]:
# Cloud Resource Allocation (Transportation Problem)
# ---------------------------------------------------
# Flow allocation from data centers to workload regions

from optilab.aliases import Model, GRB, quicksum
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem data (Cloud system structure)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(7)

n_data_centers = rng.integers(4, 8)     # supply nodes (cloud regions)
n_workloads = rng.integers(6, 12)       # demand nodes (user regions)

# -------------------------------------------------------------------------------
# Supply (compute capacity at each data center)
# -------------------------------------------------------------------------------
supply = rng.integers(50, 120, size=n_data_centers)

# -------------------------------------------------------------------------------
# Demand (compute demand from workloads)
# Constructed to GUARANTEE feasibility
# -------------------------------------------------------------------------------
raw_demand = rng.integers(20, 80, size=n_workloads)

# Scale demand so total matches supply exactly
total_supply = supply.sum()
demand = (raw_demand / raw_demand.sum() * total_supply).astype(int)

# Fix rounding drift to ensure exact balance
demand[-1] += total_supply - demand.sum()

print(f"Backend:      {m.backend_name}")
print(f"Data centers: {n_data_centers}")
print(f"Workloads:    {n_workloads}")
print(f"Total supply: {supply.sum()}")
print(f"Total demand: {demand.sum()}")

# -------------------------------------------------------------------------------
# Cost matrix (latency / bandwidth cost proxy)
# -------------------------------------------------------------------------------
# lower is better (closer data center → lower latency)
cost = rng.integers(1, 20, size=(n_data_centers, n_workloads))

print("Cost matrix:\n", cost)

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
# x[i][j] = amount of compute allocated from data center i to workload j
x = [
    [m.add_continuous_var(name=f"x_{i}_{j}") for j in range(n_workloads)]
    for i in range(n_data_centers)
]

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# Supply constraints: each data center sends all available capacity
for i in range(n_data_centers):
    m.add_constraint(
        quicksum(x[i][j] for j in range(n_workloads)) == supply[i]
    )

# Demand constraints: each workload receives exactly its required compute
for j in range(n_workloads):
    m.add_constraint(
        quicksum(x[i][j] for i in range(n_data_centers)) == demand[j]
    )

# -------------------------------------------------------------------------------
# Objective: minimize total latency / routing cost
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(
        cost[i][j] * x[i][j]
        for i in range(n_data_centers)
        for j in range(n_workloads)
    ),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
solution = np.array([[x[i][j].x for j in range(n_workloads)]
                     for i in range(n_data_centers)])

print("Optimal transport matrix:\n", solution)

print("Total cost:",
      float(np.sum(solution * cost)))

# -------------------------------------------------------------------------------
# Visualization (optional but useful for intuition)
# -------------------------------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.imshow(solution, cmap="Blues")
plt.colorbar(label="Compute allocation")
plt.title("Cloud Resource Allocation (Transportation Solution)")
plt.xlabel("Workloads")
plt.ylabel("Data Centers")
plt.show()